In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
import matplotlib.pyplot as plt

import numpy as np
import numba
from scipy.ndimage import binary_erosion

Testing random snippets of code to figure out what they do.

Sliding window tricks

In [12]:
array = np.arange(1, 10+1)
L = len(array)
l_view = 4
print("array: ", array)
slid_win = np.lib.stride_tricks.sliding_window_view(array, l_view)
print("sliding window view: \n", slid_win)

# for periodic sliding windows
array_dupe = np.tile(array, 2)
slid_win_periodic = np.lib.stride_tricks.sliding_window_view(array_dupe, l_view)[:L]
print("periodic sliding window view: \n", slid_win_periodic)

array:  [ 1  2  3  4  5  6  7  8  9 10]
sliding window view: 
 [[ 1  2  3  4]
 [ 2  3  4  5]
 [ 3  4  5  6]
 [ 4  5  6  7]
 [ 5  6  7  8]
 [ 6  7  8  9]
 [ 7  8  9 10]]
periodic sliding window view: 
 [[ 1  2  3  4]
 [ 2  3  4  5]
 [ 3  4  5  6]
 [ 4  5  6  7]
 [ 5  6  7  8]
 [ 6  7  8  9]
 [ 7  8  9 10]
 [ 8  9 10  1]
 [ 9 10  1  2]
 [10  1  2  3]]


number display

In [6]:
t = 123456
s = f"snapshot_t{t:020d}.png"
print(s)

snapshot_t00000000000000123456.png


In [1]:
from numba import njit
import numpy as np

@njit
def set_seed(seed):
    np.random.seed(seed)

@njit
def simulate(n):
    out = np.empty(n)
    for i in range(n):
        out[i] = np.random.random()
    return out

# Python side
set_seed(123)          # seed Numba’s RNG state
x1 = simulate(5)
set_seed(123)          # same seed → same results
x2 = simulate(5)

print(x1)
print(x2)              # x1 and x2 are identical

[0.69646919 0.28613933 0.22685145 0.55131477 0.71946897]
[0.69646919 0.28613933 0.22685145 0.55131477 0.71946897]


In [6]:
def local_slopes(x, y, s=1):
    y_dim = len(y)
    ms = np.zeros(y_dim - s)
    xs = np.zeros(y_dim - s)

    for i in range(y_dim - s):
        ms[i] = (y[i+s] - y[i]) / (x[i+s] - x[i])
        xs[i] = (x[i+s] + x[i])/2

    return xs, ms